# Package eclipse-nn tutorial

Package ***eclipse-nn*** is for strict, accurate, efficient, and scalable $\ell_2$-norm Lipschitz estimates on FNN. This notebook is a brief demonstration.

`LipConstEstimator` is a wrapper: you give it a network (in a few supported formats), then you call `estimate(method=...)` to run a specific method. All methods return a **certified upper bound** on the global $\ell_2$ Lipschitz constant of the network. Smaller value = tighter bound.

## Learning objectives
By the end of this notebook, you will be able to:
- Use `LipConstEstimator` as a wrapper that accepts a network
- Run different **methods** (`trivial`, `ECLipsE_Fast`, `ECLipsE`) through the same API and obtain the returned value as a certified **upper bound** on the global $\ell_2$ norm Lipschitz constant
- Verify and check: weight shapes, device/dtype mismatches, unsupported model components, and missing weight files, etc.

## Installation

**Note:** The `!pip` cell is for notebooks (Colab/Jupyter).  
If you install locally, use `pip install eclipse-nn` in your terminal.


In [ ]:
!pip install eclipse-nn

## Import

In [2]:
import torch.nn as nn
import numpy as np
import torch
import os

from eclipse_nn.LipConstEstimator import LipConstEstimator

## Create Lipschitz estimator
When creating `LipConstEstimator`, there are three ways to provide a network:
- **Torch model:** pass a PyTorch `nn.Module`
- **Given weights:** pass a list of weight tensors `[W1, W2, ..., WL]`
- **Nothing:** create an empty estimator and let the package generate a FNN


### Model 1: create estimator by torch model

In [3]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 64)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.act2 = nn.Tanh()
        self.fc3 = nn.Linear(32, 2)
        
    def forward(self, x):
        x = self.act1(self.fc1(x))
        x = self.act2(self.fc2(x))
        return self.fc3(x)

# torch model
model_t = SimpleNet()

In [4]:
# create estimator with the model
est_t = LipConstEstimator(model=model_t)
# show model information (optional)
est_t.model_review()

MODEL INFO
Layer #0: input size = 64, output size = 64, activation = ReLU.
Layer #1: input size = 64, output size = 32, activation = Tanh.
Layer #2: input size = 32, output size = 2, activation = None.
REMARK: ONLY FULLY CONNECT NEURAL NETS ARE APPLICABLE IN THIS ESTIMATOR.


### Model 2: create estimator by given weights
**Format notes:** the `.npz` file is expected to contain arrays named `w1`, `w2`, ...  
Each `wi` should have shape `[out, in]`.

In [5]:
import requests
import numpy as np
import torch

# Raw URL of weights.npz (replace with your actual raw link)
url = "https://raw.githubusercontent.com/YuezhuXu/ECLipsE/498af938ce64b5a0c9da9864a90ba43ef1afcb9c/sampleweights/npz/lyr2n80test1.npz"

# Download and save the file locally
response = requests.get(url)
response.raise_for_status()
with open("weights.npz", "wb") as f:
    f.write(response.content)

# Load the npz
weights_npz = np.load("weights.npz")

In [6]:
weights = []
wkeys = sorted(
    [k for k in weights_npz.files if k.startswith("w")],
    key=lambda k: int(k[1:])  # numeric sort by suffix
)

for k in wkeys:
    Wi = torch.tensor(weights_npz[k], dtype=torch.float64)
    weights.append(Wi)

est_w = LipConstEstimator(weights=weights)

### Model 3: create estimator for randomly generated FNN

In [7]:
est_n = LipConstEstimator()
# assign layer information (generate a FNN with layer sizes: 10 → 40 → 3)
est_n.generate_random_weights([10,40,3])

## Choose a estimation method for all the three models
Options: `trivial` (multiplication of layer norms, usually loose), `ECLipsE_Fast` (very fast and tighter), `ECLipsE` (heavier but usually tightest).

**Takeaway:** use `ECLipsE_Fast` when you want a quick certified bound; use `ECLipsE` when you want the tightest bound and can afford extra computation time.


### Model 1

In [8]:
lip_trivial_t = est_t.estimate(method='trivial')
lip_fast_t = est_t.estimate(method='ECLipsE_Fast')
lip_acc_t = est_t.estimate(method='ECLipsE') 

# show results
print('For estimator created by a torch model.')
print(f'Trivial Lipschitz estimate = {lip_trivial_t}')
print(f'Lipschitz estimate from EClipsE Fast = {lip_fast_t}')
print(f'Lipschitz estimate from EClipsE = {lip_acc_t}')

# relative tightness
print(f'Ratio Fast = {lip_fast_t / lip_trivial_t}')
print(f'Ratio Acc = {lip_acc_t / lip_trivial_t}')

Optimization reached a boundary solution. Falling back to ECLipsE-Fast for the current stage.
For estimator created by a torch model.
Trivial Lipschitz estimate = 12.615255171314553
Lipschitz estimate from EClipsE Fast = 0.5187749422351166
Lipschitz estimate from EClipsE = 0.5187749422351166
Ratio Fast = 0.04112282591118277
Ratio Acc = 0.04112282591118277


### Model 2

In [9]:
lip_trivial_w = est_w.estimate(method='trivial')
lip_fast_w = est_w.estimate(method='ECLipsE_Fast')
lip_acc_w = est_w.estimate(method='ECLipsE') 

# show results
print("For estimator created by given weights.")
print(f'Trivial Lipschitz estimate = {lip_trivial_w}')
print(f'Lipschitz estimate from EClipsE Fast = {lip_fast_w}')
print(f'Lipschitz estimate from EClipsE = {lip_acc_w}')

2
# relative tightness
print(f'Ratio Fast = {lip_fast_w / lip_trivial_w}')
print(f'Ratio Acc = {lip_acc_w / lip_trivial_w}')

For estimator created by given weights.
Trivial Lipschitz estimate = 0.9676370477588296
Lipschitz estimate from EClipsE Fast = 0.8090351972788797
Lipschitz estimate from EClipsE = 0.7372569813355624
Ratio Fast = 0.8360936563484295
Ratio Acc = 0.7619147934064154


### Model 3

In [10]:
lip_trivial_n = est_n.estimate(method='trivial')
lip_fast_n = est_n.estimate(method='ECLipsE_Fast')
lip_acc_n = est_n.estimate(method='ECLipsE') 

# show results
print('For estimator created by nothing.')
print(f'Trivial Lipschitz estimate = {lip_trivial_n}')
print(f'Lipschitz estimate from EClipsE Fast = {lip_fast_n}')
print(f'EClipsE Lip Const = {lip_acc_n}')

# relative tightness
print(f'Ratio Fast = {lip_fast_n / lip_trivial_n}')
print(f'Ratio Acc = {lip_acc_n / lip_trivial_n}')

Optimization reached a boundary solution. Falling back to ECLipsE-Fast for the current stage.
For estimator created by nothing.
Trivial Lipschitz estimate = 73.28980305832803
Lipschitz estimate from EClipsE Fast = 57.55904418078786
EClipsE Lip Const = 57.55904418078786
Ratio Fast = 0.7853622438441978
Ratio Acc = 0.7853622438441978
